In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import os
from google.colab import userdata
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
hf_token = userdata.get('HF_LOGIN')

model_google = "google/gemma-3-1b-it"
model_microsoft = "microsoft/Phi-4-mini-instruct"
#model_ministral = "ministral/Ministral-3b-instruct"

path_toxic = "toxic.csv"

# IMDb dataset download
path_movies = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("IMDb path:", path_movies)
print("\nFiles in IMDb dataset folder:")
print(os.listdir(path_movies))

# --- Load CSVs ---
df_toxic = pd.read_csv(path_toxic, on_bad_lines='skip', engine='python')
df_movies = pd.read_csv(os.path.join(path_movies, "IMDB Dataset.csv"))

# --- Preview ---
print("\n Toxic sample:")
display(df_toxic.head())

print("\n IMDb Movie Reviews sample:")
display(df_movies.head())


Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
IMDb path: /kaggle/input/imdb-dataset-of-50k-movie-reviews

Files in IMDb dataset folder:
['IMDB Dataset.csv']

 Toxic sample:


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0



 IMDb Movie Reviews sample:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df_toxic = df_toxic.drop(['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'], axis=1)
map_sentiment = {"positive" : 1, "negative" : 0}

def sample_balanced(df, label_col, n_total=500, random_state=42):
    """
    Sample n_total rows evenly across label classes.
    """

    n_classes = df[label_col].nunique()
    n_per_class = n_total // n_classes

    # Sample evenly
    df_balanced = (
        df.groupby(label_col, group_keys=False)
          .apply(lambda x: x.sample(n=min(len(x), n_per_class), random_state=random_state))
          .reset_index(drop=True)
          .sample(frac=1, random_state=random_state)
          .reset_index(drop=True)
    )

    print(f"Sampled {len(df_balanced)} rows ({n_per_class} per class).")
    return df_balanced

df_toxic_balanced = sample_balanced(df_toxic, label_col='toxic', n_total=150)
df_movies_balanced = sample_balanced(df_movies, label_col='sentiment', n_total=150)
df_movies_balanced["sentiment"] = df_movies_balanced["sentiment"].map(map_sentiment)


# --- Preview ---
print("\n Toxic Balanced Sample:")
display(df_toxic_balanced.head())

print("\n IMDb Balanced Sample:")
display(df_movies_balanced.head())

Sampled 150 rows (75 per class).
Sampled 150 rows (75 per class).

 Toxic Balanced Sample:


/tmp/ipython-input-3199460193.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_per_class), random_state=random_state))
/tmp/ipython-input-3199460193.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_per_class), random_state=random_state))


,id,comment_text,toxic
0,a66a5e12b6b108a6,"CalendarWatcher is a pedophile, child molester...",0
1,6ddd3c947fbd4d8d,== \n yeah i think so tooo \n ==,0
2,f7be51022c3e60bf,How rude \n\nSeriously? My sillypeppymacspeed ...,1
3,6ae4580294f2d587,"Editing Wikipedia \n\nFuck You, you anal rapist",1
4,3a0757419577668d,Terrorize \nI will terrorise you for as long a...,1



 IMDb Balanced Sample:


,review,sentiment
0,Well I just paid a dollar for a DVD of this mo...,0
1,I was looking forward to Kathryn Bigelow's mov...,0
2,When i first saw the movie being advertised i ...,1
3,What is there to say about an anti-establishme...,1
4,For a long time it seemed like all the good Ca...,1


In [6]:
query_qst_toxic = (
    "You are a text classification assistant.\n"
    "Classify the following text as 'toxic' or 'non-toxic'.\n"
    "Respond with exactly one word: toxic or non-toxic.\n\n"
    "Text:\n"
)


query_qst_movies = (
    "You are a sentiment classification assistant.\n"
    "Classify the following movie review as 'positive' or 'negative'.\n"
    "Respond with exactly one word: positive or negative.\n\n"
    "Review:\n"
)


def ask_LLM_abt_dataset(model_name, query_qst, dataset, text_col="review",
                        hf_token=None, top_k=20, max_new_tokens=15):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(model_name, token=hf_token).to(device)
    model.eval()

    dataset = dataset.copy()
    dataset["EPR"] = 0.0
    dataset["LLM_response"] = ""
    dataset["output_length"] = 0

    for i in range(len(dataset)):
        prompt = query_qst + dataset[text_col].iloc[i] + "\nAnswer:"
        inputs = tokenizer(
              prompt,
              return_tensors="pt",
              truncation=True,
              max_length=2048,  # avoids warning
              padding=False
        ).to(device)


        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1,
            top_p=0.9,
            return_dict_in_generate=True,
            output_scores=True
        )

        generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        # Clean and normalize response
        generated_text = (
            generated_text.strip()
                           .lower()
                           .replace(".", "")
                           .replace("**", "")
                           .split()[0] if generated_text.strip() else "unknown"
        )

        total_ent = 0.0
        for score in outputs.scores:
            probs = F.softmax(score, dim=-1)
            top_probs, _ = torch.topk(probs, k=top_k)
            safe_probs = torch.clamp(top_probs, min=1e-9)
            ent = -(safe_probs * torch.log2(safe_probs)).sum().item()
            total_ent += ent

        if len(outputs.scores) > 0:
            dataset.at[i, "EPR"] = total_ent / len(outputs.scores)
            dataset.at[i, "output_length"] = len(outputs.scores)

        dataset.at[i, "LLM_response"] = generated_text

        if i % 10 == 0:
            print(f"Processed {i+1}/{len(dataset)}")

    return dataset



In [ ]:
df_toxic_google = ask_LLM_abt_dataset(
    model_name=model_google,
    query_qst=query_qst_toxic,
    dataset=df_toxic_balanced,
    text_col="comment_text"
)

df_movies_google = ask_LLM_abt_dataset(
    model_name=model_google,
    query_qst=query_qst_movies,
    dataset=df_movies_balanced,
    text_col="review"
)

df_toxic_microsoft = ask_LLM_abt_dataset(
    model_name=model_microsoft,
    query_qst=query_qst_toxic,
    dataset=df_toxic_balanced,
    text_col="comment_text"
)

df_movies_microsoft = ask_LLM_abt_dataset(
    model_name=model_microsoft,
    query_qst=query_qst_movies,
    dataset=df_movies_balanced,
    text_col="review"
)

"""
df_toxic_ministral = ask_LLM_abt_dataset(
    model_name=model_ministral,
    query_qst=query_qst_toxic,
    dataset=df_toxic_balanced,
    text_col="comment_text"
)

df_movies_ministral = ask_LLM_abt_dataset(
    model_name=model_ministral,
    query_qst=query_qst_movies,
    dataset=df_movies_balanced,
    text_col="review"
)
"""

Processed 1/150
Processed 11/150
Processed 21/150
Processed 31/150
Processed 41/150
Processed 51/150
Processed 61/150
Processed 71/150
Processed 81/150
Processed 91/150
Processed 101/150
Processed 111/150
Processed 121/150
Processed 131/150
Processed 141/150
Processed 1/150
Processed 11/150
Processed 21/150
Processed 31/150
Processed 41/150
Processed 51/150
Processed 61/150
Processed 71/150
Processed 81/150
Processed 91/150
Processed 101/150
Processed 111/150
Processed 121/150
Processed 131/150
Processed 141/150


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
df_movies_google.to_csv("df_movies_google.csv", index=False)
df_movies_microsoft.to_csv("df_movies_microsoft.csv", index=False)
#df_movies_ministral.to_csv("df_movies_ministral.csv", index=False)
df_toxic_google.to_csv("df_toxic_google.csv", index=False)
df_toxic_microsoft.to_csv("df_toxic_microsoft.csv", index=False)
#df_toxic_ministral.to_csv("df_toxic_ministral.csv", index=False)

In [ ]:
def calculate_metrics(df, label_col = "toxic") :

  # true labels (0 = non-toxic, 1 = toxic)
  y_true = df[label_col]

  # model scores (EPR values)
  y_scores = df["EPR"]

  # ROC-AUC (area under the ROC curve)
  roc_auc = roc_auc_score(y_true, y_scores)

  # PR-AUC (area under the Precision-Recall curve)
  pr_auc = average_precision_score(y_true, y_scores)

  print(f"ROC-AUC: {roc_auc:.4f}")
  print(f"PR-AUC:  {pr_auc:.4f}")


In [ ]:
calculate_metrics(df_toxic_google, label_col="toxic")

In [ ]:
calculate_metrics(df_toxic_microsoft, label_col="toxic")

In [ ]:
calculate_metrics(df_movies_google, label_col="sentiment")

In [ ]:
calculate_metrics(df_movies_microsoft, label_col="sentiment")